In [1]:
import os
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)

'socks://127.0.0.1:7897/'

In [1]:
pip install open_clip_torch

In [13]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [16]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16-quickgelu', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16-quickgelu')

In [5]:
mkdir -p features data scripts src

In [5]:
from clip_zeroshot import build_and_cache_image_features, load_cached_image_features, load_cached_text_features

## Run imagenet-R script

In [ ]:
!python -u scripts/eval_imagenet_r.py --data-root ../data

loading axiong/imagenet-r
loading cached image features from /content/features/r_few_shot_image_features.pt
loading cached text features from /content/features/r_text-features.pt
loading cached image features from /content/features/r_eval_features.pt
CoOp row skipped: no context vector (trained by train_coop_imagenet_r.py)
running TPT (per-image gradient steps, this is slow)
  0%|          | 0/200 [00:00<?, ?it/s]
TPT mean confidence: 0.672893226146698
TPT min/max confidence: 0.051550768315792084 0.9999622106552124

zero-shot   : top-1 77.67%  ECE 3.79% Signed-Gap -3.78% (n=26800)
Tip-Adapter : top-1 78.52%  ECE 0.88% Signed-Gap 0.68% (n=26800)
TPT         : top-1 72.00%  ECE 7.23% Signed-Gap -4.71% (n=200)


In [ ]:
!pip install --upgrade 'gdown>=6.2.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


## Run imagenet-Sketch script

In [ ]:
!gdown --cookies drive.google.com_cookies.txt https://drive.google.com/uc?id=1Mj0i5HBthqH1p_yeXzsg22gZduvgoNeA

Using cookies from drive.google.com_cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1Mj0i5HBthqH1p_yeXzsg22gZduvgoNeA
From (redirected): https://drive.google.com/uc?id=1Mj0i5HBthqH1p_yeXzsg22gZduvgoNeA&confirm=t&uuid=cf7cc89b-b309-4fc0-b6b0-849060909db5&at=AAUTciQePl07B_lL2bHOXYE9VEk_%3A1790050881983
To: /content/ImageNet-Sketch.zip
100% 7.59G/7.59G [03:04<00:00, 41.1MB/s]


In [ ]:
!mv ImageNet-Sketch.zip ./data

In [ ]:
!python -u scripts/eval_imagenet_sketch.py --data-root ./data

extracting data/ImageNet-Sketch.zip
/content/features/sk_few_shot_image_features.pt not found, extracting image features (this needs a GPU)
  0%|          | 0/500 [00:00<?, ?it/s]
Image features and labels has been saved at /content/features/sk_few_shot_image_features.pt
/content/features/sk_eval_features.pt not found, extracting image features (this needs a GPU)
  0%|          | 0/1091 [00:00<?, ?it/s]
Image features and labels has been saved at /content/features/sk_eval_features.pt
/content/features/sk_text-features.pt not found, building text features
  0%|          | 0/1000 [00:00<?, ?it/s]
Text features has been saved at /content/features/sk_text-features.pt

zero-shot   : top-1 48.44%  ECE 4.33% Signed-Gap 4.33% (n=34889)
Tip-Adapter : top-1 55.83%  ECE 12.00% Signed-Gap 12.00% (n=34889)


## Run Zeroshot on r and sk200

### Build the R's features

In [6]:
r_all_features = load_cached_image_features('/content/features/r_all_features.pt')

assert r_all_features["image_features"].shape[0] == 30_000

### Build the Sk200's features

In [9]:
if sys.modules['sk200_all_labels']:
  del sys.modules['sk200_all_labels']

KeyError: 'sk200_all_labels'

In [7]:
from sk200_all_labels import sk200_lables

In [8]:
sk200_all_features = load_cached_image_features('/content/features/sk200_all_features.pt')

### Run the harness on both R and sk200's test split

In [21]:
if sys.modules['splits']:
  del sys.modules['splits']

In [9]:
from splits import split_indices
from harness import run_comparison, zero_shot_logits, accuracy, ece, signed_gap

text_features = load_cached_text_features('/content/features/r_text-features.pt')
metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}
methods = {"zero_shot": {"fn": zero_shot_logits, "params": {}}}

In [10]:
text_features = text_features['text_features']

In [13]:
d = {
    "r_all_features" : r_all_features,
    'sk200_all_features': sk200_all_features
}

for name, feats in d.items():
    cache_idx, val_idx, test_idx = split_indices(feats['labels'].tolist(), seed=42)

    shared = {
        "test_features": feats["image_features"][test_idx].to(device),
        "labels": feats["labels"][test_idx].to(device),
        "text_features": text_features.to(device),
        "logit_scale": model.logit_scale.exp(),
    }

    print(name, len(test_idx), run_comparison(shared, methods, metrics))

r_all_features 24800 {'zero_shot': {'accuracy': 77.8104841709137, 'ece': 3.8871455937623978, 'signed_gap': -3.869563341140747}}
sk200_all_features 4952 {'zero_shot': {'accuracy': 82.10824131965637, 'ece': 2.9883261770009995, 'signed_gap': -2.869904041290283}}


In [14]:
ev = load_cached_image_features('/content/features/r_eval_features.pt')
shared = {
    "test_features": ev["image_features"].to(device),
    "labels": ev["labels"].to(device),
    "text_features": text_features.to(device),
    "logit_scale": model.logit_scale.exp(),
}
print(run_comparison(shared, methods, metrics))

{'zero_shot': {'accuracy': 77.6716411113739, 'ece': 3.790641948580742, 'signed_gap': -3.780156373977661}}


In [15]:
print(torch.equal(r_all_features["image_features"][3200:], ev["image_features"]),
      torch.equal(r_all_features["labels"][3200:], ev["labels"]))

True True


In [16]:
shared = {
    "test_features": r_all_features["image_features"].to(device),
    "labels": r_all_features["labels"].to(device),
    "text_features": text_features.to(device),
    "logit_scale": model.logit_scale.exp(),
}
print(run_comparison(shared, methods, metrics))

{'zero_shot': {'accuracy': 77.46000289916992, 'ece': 3.6926954984664917, 'signed_gap': -3.6812543869018555}}


In [17]:
r_few_shot_cache = load_cached_image_features('/content/features/r_few_shot_image_features.pt')

In [18]:
shared = {
    "test_features": r_few_shot_cache["image_features"].to(device),
    "labels": r_few_shot_cache["labels"].to(device),
    "text_features": text_features.to(device),
    "logit_scale": model.logit_scale.exp(),
}
print(run_comparison(shared, methods, metrics))

{'zero_shot': {'accuracy': 71.49999737739563, 'ece': 2.6339836418628693, 'signed_gap': -2.4867236614227295}}


## Run zero shot on PACS

### Download and prepare the data

In [19]:
!curl -L -o ./data/pacs-dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/nickfratto/pacs-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  525M  100  525M    0     0  51.8M      0  0:00:10  0:00:10 --:--:-- 61.4M


In [25]:
!unzip -qq ../data/pacs-dataset.zip

/content


In [26]:
domains = ["photo", "art_painting", "cartoon", "sketch"]

pacs_datasets = {
    domain: datasets.ImageFolder(root=f"./pacs_data/pacs_data/{domain}", transform=preprocess)
    for domain in domains
}

pacs_dataloaders = {
    domain: DataLoader(pacs_datasets[domain], batch_size=32, shuffle=False, num_workers=2)
    for domain in domains
}

In [27]:
for domain in domains:
    print(domain, pacs_datasets[domain].classes)

photo ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
art_painting ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
cartoon ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
sketch ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


In [28]:
for domain in domains:
    print(domain, pacs_datasets[domain].class_to_idx)

photo {'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}
art_painting {'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}
cartoon {'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}
sketch {'dog': 0, 'elephant': 1, 'giraffe': 2, 'guitar': 3, 'horse': 4, 'house': 5, 'person': 6}


In [29]:
mappings = [pacs_datasets[d].class_to_idx for d in domains]
all_match = all(m == mappings[0] for m in mappings)
print(all_match)

True


### Build the text features

In [30]:
pacs_classes = [cls for cls in pacs_datasets['photo'].classes]
print(pacs_classes)

['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


In [33]:
from imagenet_classes import IMAGENET_TEMPLATES
from clip_zeroshot import build_and_cache_text_features

pacs_text_features = build_and_cache_text_features(model, tokenizer, pacs_classes, IMAGENET_TEMPLATES, device, './features', 'pacs_text_features')

  0%|          | 0/7 [00:00<?, ?it/s]

Text features has been saved at ./features/pacs_text_features.pt


In [36]:
type(pacs_text_features)

torch.Tensor

In [40]:
for domain in domains:
    cached = build_and_cache_image_features(model, device, pacs_dataloaders[domain], "./features", f"pacs_{domain}")

    pacs_image_features = cached['image_features']
    pacs_labels = cached['labels']

    shared = {
        "test_features": pacs_image_features.to(device),
        "labels": pacs_labels.to(device),
        "text_features": pacs_text_features.to(device),
        "logit_scale": model.logit_scale.exp(),
    }

    print(run_comparison(shared, methods, metrics))

  0%|          | 0/53 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/pacs_photo.pt
{'zero_shot': {'accuracy': 99.94012117385864, 'ece': 1.2024708092212677, 'signed_gap': -1.202470064163208}}


  0%|          | 0/64 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/pacs_art_painting.pt
{'zero_shot': {'accuracy': 97.65625, 'ece': 2.6192616671323776, 'signed_gap': -2.3866891860961914}}


  0%|          | 0/74 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/pacs_cartoon.pt
{'zero_shot': {'accuracy': 99.274742603302, 'ece': 2.0681703463196754, 'signed_gap': -2.0681679248809814}}


  0%|          | 0/123 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/pacs_sketch.pt
{'zero_shot': {'accuracy': 90.20106792449951, 'ece': 3.024270199239254, 'signed_gap': 3.015536069869995}}


#### Test load_features function

In [18]:
from features_registry import load_features
from splits import split_indices
from harness import run_comparison, zero_shot_logits, accuracy, ece, signed_gap

metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}
methods = {"zero_shot": {"fn": zero_shot_logits, "params": {}}}

for name in ["imagenet_r", "sketch_200"]:
    f = load_features(name, device=device)
    cache_idx, val_idx, test_idx = split_indices(f["labels"].tolist(), seed=42)

    shared = {
        "test_features": f["image_features"][test_idx],
        "labels": f["labels"][test_idx].to(device),
        "text_features": f["text_features"],
        "logit_scale": model.logit_scale.exp(),
    }
    print(name, len(test_idx), run_comparison(shared, methods, metrics))

imagenet_r 24800 {'zero_shot': {'accuracy': 77.8104841709137, 'ece': 3.8871347904205322, 'signed_gap': -3.869551420211792}}
sketch_200 4952 {'zero_shot': {'accuracy': 82.10824131965637, 'ece': 2.9883189126849174, 'signed_gap': -2.869904041290283}}


#### Test the run_grid script

In [2]:
pwd

'/home/lcoach/research/vlm-reliability/notebooks'

In [5]:
!python3 ../scripts/run_grid.py

imagenet_r    seed 42  ZS gap -3.87  Δβ5 +4.57  Δβ1 +3.85
imagenet_r    seed 43  ZS gap -3.77  Δβ5 +4.35  Δβ1 +3.47
imagenet_r    seed 44  ZS gap -3.77  Δβ5 +4.47  Δβ1 +3.74
sketch_200    seed 42  ZS gap -2.87  Δβ5 +5.15  Δβ1 +2.88
sketch_200    seed 43  ZS gap -2.68  Δβ5 +4.95  Δβ1 +3.17
sketch_200    seed 44  ZS gap -2.22  Δβ5 +5.16  Δβ1 +3.15
sketch_1000   seed 42  ZS gap +4.48  Δβ5 +7.49  Δβ1 +4.44
sketch_1000   seed 43  ZS gap +4.59  Δβ5 +7.42  Δβ1 +4.43
sketch_1000   seed 44  ZS gap +4.69  Δβ5 +7.75  Δβ1 +4.37
imagenet_v2   seed 42  ZS gap +1.97  Δβ5 +2.40  Δβ1 +1.57
imagenet_v2   seed 43  ZS gap +1.94  Δβ5 +2.33  Δβ1 +1.88
imagenet_v2   seed 44  ZS gap +2.01  Δβ5 +2.24  Δβ1 +1.82
pacs_photo    seed 42  ZS gap -1.24  Δβ5 +1.22  Δβ1 +1.06
pacs_photo    seed 43  ZS gap -1.23  Δβ5 +1.24  Δβ1 +1.21
pacs_photo    seed 44  ZS gap -1.19  Δβ5 +1.16  Δβ1 +1.12
pacs_art      seed 42  ZS gap -2.44  Δβ5 +1.53  Δβ1 +1.39
pacs_art      seed 43  ZS gap -2.31  Δβ5 +1.31  Δβ1 +0.80
pacs_art      